In [6]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

#初始化模型
model = init_chat_model("gpt-4o")

## 1、工具调用

In [7]:
# 模拟工具
from langchain.tools import tool
@tool
def create_calendar_event(event_name: str, event_time: str):
    """根据事件名称和具体日期创建一个日程安排"""
    return f"创建了一个名为 {event_name} 的日程，时间为 {event_time}"

@tool
def send_email(to: str, subject: str, body: str):
    """发送一封邮件"""
    return f"发送了一封邮件给 {to}，主题为 {subject}，内容为 {body}"

In [8]:
# 1、子智能体、日历智能体：日程安排
CALENDAR_PROMPT = """
你是一个日程安排助手，请根据用户的需求，创建日程安排。
使用以下工具：
- 创建日程安排：create_calendar_event(event_name, event_time)
"""

calendar_agent = create_agent(
    model=model,
    tools=[create_calendar_event],
    system_prompt=CALENDAR_PROMPT,
)

# 2、子智能体、邮件智能体：邮件处理

EMAIL_PROMPT = """
你是一个邮件助手，请根据用户的需求，提取发送邮件的信息，并撰写适当的主题和正文，发送邮件。
使用以下工具：
- 发送邮件：send_email(to, subject, body)
"""

email_agent = create_agent(
    model=model,
    tools=[send_email],
    system_prompt=EMAIL_PROMPT,
)

In [9]:
from langgraph.checkpoint.memory import InMemorySaver

MAIN_PROMPT = """
你是一个日程安排和邮件助手，请根据用户的需求，创建日程安排，撰写邮件并发送。
使用以下工具：
- 日历智能体：{calendar_agent}
- 邮件智能体：{email_agent}
"""

@tool
def calendar_event(request:str):
    """根据用户的请求和安排事件，创建日程安排"""
    result = calendar_agent.invoke(
        {"messages":[
            {"role": "user", "content": request}
        ]}
    )
    #将子智能体的结果作为当前工具的消息返回给主智能体
    return f"该事件的日程安排为：{result["messages"][-1].content}"

@tool
def email_event(request:str):
    """根据用户的请求和安排事件，创建邮件并发送"""
    result = email_agent.invoke(
        {"messages":[
            {"role": "user", "content": request}
        ]}
    )
    return f"根据事件已经完成邮件发送：{result["messages"][-1].content}"


main_agent = create_agent(
    model=model,
    tools=[calendar_event, email_event],
    system_prompt=MAIN_PROMPT,
    checkpointer=InMemorySaver()
)

In [10]:
request= "请帮我安排一下会议，下周二下午3点，主题为‘项目进度汇报’，并发送邮件给参会人员，具体为：张三（zhangsan@example.com）、李四(lisi@example.com)、王五(wangwu@example.com)"
config = {"configurable":{"thread_id":"user-001"}}
result = main_agent.invoke({
    "messages":[
        {"role": "user", "content": request}
    ]
},config=config)

In [11]:
print(result["messages"][-1].content)

会议“项目进度汇报”已成功安排，时间为下周二下午3点。同时，会议通知邮件也已发送给以下参会人员：

1. 张三（zhangsan@example.com）
2. 李四（lisi@example.com）
3. 王五（wangwu@example.com）

如有其他需要，请随时告知！


## 2、交接模式

In [12]:
# Handoffs核心机制：工具会更新一个状态变量，系统读取这个变量来动态调整Agent的行为（System_prompt, 切换可用工具），或者直接将控制权路由给另一个Agent

from langchain.agents import AgentState

class SupportState(AgentState):
    """跟踪当前处理哪个步骤"""
    current_step: str = "triage" # 默认初始状态为：信息收集
    warranty_status: str | None = None # 用于存储收集到的保修状态


In [13]:
# 提供一个工具，用来修改状态
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages  import ToolMessage

@tool
def record_warranty_status(status: str, runtime:ToolRuntime)->Command:
    """记录保修状态并转换成下一步"""
    return Command(
        update={
            "messages": [
                ToolMessage(
                    content=f"保修状态已记录：{status}",
                    tool_call_id=runtime.tool_call_id  
                )
            ],
            "current_step": "specialist",  # 核心动作：更新状态变量，触发行为改变
            "warranty_status": status 
        }
    )

In [14]:
@tool
def provide_solution(runtime:ToolRuntime):
    """提供解决方案"""
    warranty_status = runtime.state.get("warranty_status", "")
    return f"基于你目前的保修状态：{warranty_status}，我们可以提供的方案是换个新的给你，请稍等，我们会尽快给你更换新的产品。"

@tool
def escalate(runtime:ToolRuntime):
    """升级问题到技术支持"""
    return "我们将升级问题到技术支持团队，请稍等。。。。。。"
    

In [15]:
# 当前Agent如何切换状态system_prompt|tools
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def apply_step_config(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """根据current_stemp动态配置Agent的行为"""
    step = request.state.get("current_step", "triage")

    # 定义不同状态的配置映射
    configs = {
        "triage": {
            "prompt": "你是一个客服接待员，请只咨询用户的设备保修状态，不提供解决方案！",
            "tools": [record_warranty_status]
        },
        "specialist": {
            "prompt": f"你是一个技术支持专家。已知用户保修状态为：{request.state.get("warranty_status", "")}。请提供解决方案或者升级问题到技术支持团队。",
            "tools": [provide_solution, escalate]
        }
    }

    config = configs[step]
    
    # 核心动作：拦截并覆盖当前的请求配置
    request = request.override(
        system_prompt=config["prompt"],
        tools=config["tools"]
    )
    
    #将修改后的请求传递给实际的模型处理程序
    return handler(request)

In [16]:
agent = create_agent(
    model=model,
    tools=[record_warranty_status, provide_solution, escalate],#注册所有可能的工具
    state_schema=SupportState, #绑定我们定义的状态结构
    middleware=[apply_step_config], # 绑定中间件，切换Agent的行为
    checkpointer=InMemorySaver()
)

In [17]:
config = {"configurable":{"thread_id":"user-003"}}
result = agent.invoke(
    {
        "messages":[{"role": "user", "content": "你好，我家的电视坏了，能给我看看吗？"}]
    },config
)

In [18]:
result

{'messages': [HumanMessage(content='你好，我家的电视坏了，能给我看看吗？', additional_kwargs={}, response_metadata={}, id='933f74f1-0f4e-4f19-9238-6a1c9d59b28d'),
  AIMessage(content='您好！我可以帮助您查询电视设备的保修状态。请问您的设备目前是在保修期内还是保修已过？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 84, 'total_tokens': 115, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_ad9f0a4804', 'id': 'chatcmpl-EmCEZ490ycW4n7jbFSFBXL6ARKpso', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1f79a-c933-7372-8a57-321338610812-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 84, 'output_tokens': 31, 'total_tokens': 115, 'in

In [19]:
result2 = agent.invoke(
    {
        "messages":[{"role": "user", "content": "刚买没几天，还在保修期内！"}]
    },config
)

In [20]:
result2

{'messages': [HumanMessage(content='你好，我家的电视坏了，能给我看看吗？', additional_kwargs={}, response_metadata={}, id='933f74f1-0f4e-4f19-9238-6a1c9d59b28d'),
  AIMessage(content='您好！我可以帮助您查询电视设备的保修状态。请问您的设备目前是在保修期内还是保修已过？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 84, 'total_tokens': 115, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_ad9f0a4804', 'id': 'chatcmpl-EmCEZ490ycW4n7jbFSFBXL6ARKpso', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1f79a-c933-7372-8a57-321338610812-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 84, 'output_tokens': 31, 'total_tokens': 115, 'in

In [ ]:
print(result2["messages"][-1].content)

针对您的问题：

1. 根据保修状态，我们可为您提供解决方案：换一个新的设备给您。
2. 同时，我已将您的问题升级至技术支持团队，他们会尽快联系您，确保为您提供进一步的服务。

请保持您的联系方式畅通，感谢您的耐心等待！如有其他问题，欢迎随时联系客服。


Failed to get info from https://api.smith.langchain.com: LangSmithConnectionError('Connection error caused failure to GET /info in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError(\'HTTPSConnectionPool(host=\\\'api.smith.langchain.com\\\', port=443): Max retries exceeded with url: /info (Caused by ReadTimeoutError("HTTPSConnectionPool(host=\\\'api.smith.langchain.com\\\', port=443): Read timed out. (read timeout=10.0)"))\'))\nContent-Length: None\nAPI Key: lsv2_********************************************3e')
Run compression is not enabled. Please update to the latest version of LangSmith. Falling back to regular multipart ingestion.
Failed to multipart ingest runs: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /runs/multipart